# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [1]:
def lettre_vers_nombre(c):
    """Numéro de 0 à 25 d'une lettre majuscule."""
    return ord(c) - ord("A")

def nombre_vers_lettre(n):
    """Lettre majuscule correspondant à un numéro, ramené entre 0 et 25."""
    return chr(n % 26 + ord("A"))

print(lettre_vers_nombre("C"), nombre_vers_lettre(2))
# réciproques l'une de l'autre sur tout l'alphabet
print(all(nombre_vers_lettre(lettre_vers_nombre(chr(ord("A") + i))) == chr(ord("A") + i) for i in range(26)))

2 C
True


## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [2]:
import unicodedata

def nettoyer(texte):
    """Majuscules sans accents, lettres de A à Z uniquement."""
    # NFD sépare chaque lettre accentuée en lettre + accent (catégorie Unicode "Mn")
    decompose = unicodedata.normalize("NFD", texte)
    sans_accent = "".join(c for c in decompose if unicodedata.category(c) != "Mn")
    return "".join(c for c in sans_accent.upper() if "A" <= c <= "Z")

print(nettoyer("Été, à Paris !"))

ETEAPARIS


## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [3]:
def chiffrer(texte, a, b):
    """Chiffrement affine E(x) = (a x + b) mod 26 avec la clé (a, b)."""
    return "".join(nombre_vers_lettre(a * lettre_vers_nombre(c) + b) for c in nettoyer(texte))

print(chiffrer("ATTAQUEALAUBE", 5, 8))
print(chiffrer("ATTAQUEALAUBE", 1, 3))   # a = 1 : le chiffre de César

IZZIKECILIENC
DWWDTXHDODXEH


## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [4]:
def pgcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

# a est utilisable si et seulement si a est inversible modulo 26, c'est-à-dire pgcd(a, 26) = 1
valides = [a for a in range(1, 26) if pgcd(a, 26) == 1]
print(valides)
print(len(valides), "valeurs possibles pour a")

# 26 = 2 x 13 : on exclut les nombres pairs et 13.
# Avec les 26 valeurs de b, cela fait 12 x 26 = 312 clés en tout.
print(len(valides) * 26, "clés")

[1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
12 valeurs possibles pour a
312 clés


## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [ ]:
def inverse_modulaire(a, n):
    """Inverse de a modulo n. Lève ValueError s'il n'existe pas."""
    a = a % n
    for candidat in range(1, n):
        if a * candidat % n == 1:
            return candidat
    raise ValueError(f"{a} n'est pas inversible modulo {n}.")

print(inverse_modulaire(5, 26))
print(5 * 21 % 26)

try:
    inverse_modulaire(4, 26)
except ValueError as err:
    print("erreur attendue :", err)

21
1
erreur attendue : 4 n'est pas inversible modulo 26.


## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [6]:
def dechiffrer(texte, a, b):
    """Déchiffrement affine x = a^-1 (y - b) mod 26. Lève ValueError si la clé n'est pas inversible."""
    a_inverse = inverse_modulaire(a, 26)
    return "".join(nombre_vers_lettre(a_inverse * (lettre_vers_nombre(c) - b)) for c in nettoyer(texte))

# test aller-retour : chiffrer puis déchiffrer redonne le message nettoyé
message = nettoyer("Attaque à l'aube !")
secret = chiffrer(message, 5, 8)
print(secret)
print(dechiffrer(secret, 5, 8))
print(dechiffrer(secret, 5, 8) == message)

# l'aller-retour, sur toutes les clés valides
print(all(dechiffrer(chiffrer(message, a, b), a, b) == message for a in valides for b in range(26)))

# une clé non inversible est refusée explicitement...
try:
    dechiffrer(secret, 4, 7)
except ValueError as erreur:
    print("clé refusée :", erreur)

# ... car avec a = 4, des lettres différentes ont la même image : l'information est perdue
print(chiffrer("AN", 4, 7))

IZZIKECILIENC
ATTAQUEALAUBE
True
True
clé refusée : 4 n'est pas inversible modulo 26.
HH


## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [7]:
def toutes_les_clefs(secret):
    """Tous les déchiffrements possibles, sous forme de triplets (a, b, texte)."""
    return [(a, b, dechiffrer(secret, a, b)) for a in valides for b in range(26)]

essais = toutes_les_clefs(secret)
print(len(essais), "clés essayées")

# les 312 propositions, quatre par ligne, à parcourir à l'œil
for i in range(0, len(essais), 4):
    print("   ".join(f"{a:>2} {b:>2} {texte}" for a, b, texte in essais[i:i + 4]))

# à l'œil, une seule ligne est du français : on vérifie
bons = [(a, b) for a, b, texte in essais if texte == message]
print("la bonne clé :", bons)

312 clés essayées
 1  0 IZZIKECILIENC    1  1 HYYHJDBHKHDMB    1  2 GXXGICAGJGCLA    1  3 FWWFHBZFIFBKZ
 1  4 EVVEGAYEHEAJY    1  5 DUUDFZXDGDZIX    1  6 CTTCEYWCFCYHW    1  7 BSSBDXVBEBXGV
 1  8 ARRACWUADAWFU    1  9 ZQQZBVTZCZVET    1 10 YPPYAUSYBYUDS    1 11 XOOXZTRXAXTCR
 1 12 WNNWYSQWZWSBQ    1 13 VMMVXRPVYVRAP    1 14 ULLUWQOUXUQZO    1 15 TKKTVPNTWTPYN
 1 16 SJJSUOMSVSOXM    1 17 RIIRTNLRURNWL    1 18 QHHQSMKQTQMVK    1 19 PGGPRLJPSPLUJ
 1 20 OFFOQKIOROKTI    1 21 NEENPJHNQNJSH    1 22 MDDMOIGMPMIRG    1 23 LCCLNHFLOLHQF
 1 24 KBBKMGEKNKGPE    1 25 JAAJLFDJMJFOD    3  0 URRUMKSUVUKNS    3  1 LIILDBJLMLBEJ
 3  2 CZZCUSACDCSVA    3  3 TQQTLJRTUTJMR    3  4 KHHKCAIKLKADI    3  5 BYYBTRZBCBRUZ
 3  6 SPPSKIQSTSILQ    3  7 JGGJBZHJKJZCH    3  8 AXXASQYABAQTY    3  9 ROORJHPRSRHKP
 3 10 IFFIAYGIJIYBG    3 11 ZWWZRPXZAZPSX    3 12 QNNQIGOQRQGJO    3 13 HEEHZXFHIHXAF
 3 14 YVVYQOWYZYORW    3 15 PMMPHFNPQPFIN    3 16 GDDGYWEGHGWZE    3 17 XUUXPNVXYXNQV
 3 18 OLLOGEMOPOEHM    3 19 FCCFXVDF

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [8]:
from collections import Counter

texte_long = nettoyer(
    "Le chiffrement affine est un chiffrement par substitution monoalphabetique. "
    "Chaque lettre du message est remplacee par une autre lettre, toujours la meme, "
    "ce qui laisse intactes les frequences relatives des lettres et permet de casser "
    "le code sans essayer toutes les clefs possibles."
)
secret_long = chiffrer(texte_long, 7, 12)

# fréquences des lettres en français, en %
FREQUENCES_FR = {
    "A": 8.15, "B": 0.97, "C": 3.15, "D": 3.73, "E": 17.39, "F": 1.12, "G": 0.97,
    "H": 0.85, "I": 7.31, "J": 0.45, "K": 0.02, "L": 5.69, "M": 2.87, "N": 7.12,
    "O": 5.28, "P": 3.03, "Q": 0.99, "R": 6.55, "S": 8.08, "T": 7.07, "U": 5.74,
    "V": 1.32, "W": 0.04, "X": 0.45, "Y": 0.30, "Z": 0.12,
}

def score_francais(texte):
    """Ressemblance au français : somme des fréquences observées x fréquences attendues."""
    comptes = Counter(texte)
    return sum(comptes[c] / len(texte) * FREQUENCES_FR[c] for c in comptes)

def casser(secret):
    """Suppose que la lettre la plus fréquente du chiffré est l'image de E,
    et renvoie le candidat (a, b, texte) qui ressemble le plus au français."""
    lettre_frequente = Counter(secret).most_common(1)[0][0]
    cible = lettre_vers_nombre(lettre_frequente)
    candidats = []
    for a in valides:
        # E (numéro 4) est chiffré en cible : 4a + b = cible mod 26
        b = (cible - 4 * a) % 26
        candidats.append((a, b, dechiffrer(secret, a, b)))
    print(f"lettre la plus fréquente : {lettre_frequente}, {len(candidats)} candidats au lieu de 312")
    return max(candidats, key=lambda candidat: score_francais(candidat[2]))

a, b, texte = casser(secret_long)
print("clé trouvée :", a, b)
print(texte)
print(texte == texte_long)

lettre la plus fréquente : O, 12 candidats au lieu de 312
clé trouvée : 7 12
LECHIFFREMENTAFFINEESTUNCHIFFREMENTPARSUBSTITUTIONMONOALPHABETIQUECHAQUELETTREDUMESSAGEESTREMPLACEEPARUNEAUTRELETTRETOUJOURSLAMEMECEQUILAISSEINTACTESLESFREQUENCESRELATIVESDESLETTRESETPERMETDECASSERLECODESANSESSAYERTOUTESLESCLEFSPOSSIBLES
True


In [9]:
# la même attaque sur le message court échoue
a, b, texte = casser(secret)
print("clé trouvée :", a, b, "->", texte)
print(Counter(message).most_common(3))

lettre la plus fréquente : I, 12 candidats au lieu de 312
clé trouvée : 15 0 -> ETTESCOEZECNO
[('A', 4), ('T', 2), ('U', 2)]


**Pourquoi l'attaque échoue sur un texte court.** Elle suppose que la lettre la plus fréquente du chiffré est l'image de E. C'est vrai sur un texte long, mais sur 13 lettres les comptes sont trop petits pour refléter les fréquences du français : dans `ATTAQUEALAUBE`, la lettre la plus fréquente est A (4 fois), pas E (2 fois). L'hypothèse de départ est donc fausse, aucune des 12 clés testées n'est la bonne, et le meilleur candidat reste illisible.

## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [10]:
import random

def indice_coincidence(texte):
    """Probabilité que deux lettres tirées au hasard dans le texte soient identiques."""
    N = len(texte)
    return sum(n * (n - 1) for n in Counter(texte).values()) / (N * (N - 1))

random.seed(1)
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
hasard = "".join(random.choice(alphabet) for _ in range(len(texte_long)))

print(f"clair  : {indice_coincidence(texte_long):.4f}")
print(f"affine : {indice_coincidence(secret_long):.4f}")
print(f"hasard : {indice_coincidence(hasard):.4f}")

# Le chiffrement affine permute les lettres : il permute les n_i sans changer
# leur somme, donc l'indice du chiffré est exactement celui du clair.

clair  : 0.0872
affine : 0.0872
hasard : 0.0387


## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [11]:
def vigenere(texte, cle):
    """Chiffre de Vigenère : chaque lettre est décalée du rang de la lettre de la clé en face."""
    resultat = []
    for i, c in enumerate(nettoyer(texte)):
        decalage = lettre_vers_nombre(cle[i % len(cle)])
        resultat.append(nombre_vers_lettre(lettre_vers_nombre(c) + decalage))
    return "".join(resultat)

secret_vigenere = vigenere(texte_long, "CLE")
print(secret_vigenere[:40])
print(f"indice vigenère : {indice_coincidence(secret_vigenere):.4f}")

NPGJTJHCIOPRVLJHTRGPWVFRESMHQVGXIPETCCWW
indice vigenère : 0.0549


## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [12]:
for L in range(1, 8):
    colonnes = [secret_vigenere[i::L] for i in range(L)]   # une lettre sur L, à partir de i
    moyenne = sum(indice_coincidence(col) for col in colonnes) / L
    print(f"L = {L} : indice moyen {moyenne:.4f}")

# L = 3 : chaque colonne est un César, la lettre la plus fréquente est l'image de E
L = 3
cle = ""
for i in range(L):
    lettre_frequente = Counter(secret_vigenere[i::L]).most_common(1)[0][0]
    cle += nombre_vers_lettre(lettre_vers_nombre(lettre_frequente) - lettre_vers_nombre("E"))
print("clé retrouvée :", cle)

L = 1 : indice moyen 0.0549
L = 2 : indice moyen 0.0557
L = 3 : indice moyen 0.0916
L = 4 : indice moyen 0.0539
L = 5 : indice moyen 0.0536
L = 6 : indice moyen 0.0918
L = 7 : indice moyen 0.0558
clé retrouvée : CLE


## L'IA vous le donne en trois secondes

**Chiffrement affine.** J'ai posé la question à un assistant IA, puis j'ai exécuté le code qu'il proposait pour vérifier ses affirmations.

Le code vérifie la clé : avec a = 4 et b = 7, `chiffrer` comme `dechiffrer` lèvent `ValueError`. Il calcule l'inverse avec `pow(a, -1, 26)`. Seul défaut : les lettres accentuées ne sont pas chiffrées (`"Été à Paris"` donne `"Ézé à Fipwu"`), ce qui laisse passer de l'information.